<a href="https://colab.research.google.com/github/Justinl20/Justinl20-DataScience-GenAI-Submissions/blob/main/6_02_DNN_101_COMPLETED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [ ]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [ ]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

:We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [ ]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [ ]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [ ]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [ ]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 32465.2695
Epoch [10/100], Loss: 33047.5898
Epoch [10/100], Loss: 25585.9004
Epoch [10/100], Loss: 24925.2285
Epoch [10/100], Loss: 23076.3516
Epoch [10/100], Loss: 34555.8281
Epoch [10/100], Loss: 32154.3613
Epoch [10/100], Loss: 50474.8555
Epoch [20/100], Loss: 26961.1797
Epoch [20/100], Loss: 30235.3164
Epoch [20/100], Loss: 35084.7852
Epoch [20/100], Loss: 24425.9922
Epoch [20/100], Loss: 32085.8652
Epoch [20/100], Loss: 26013.9043
Epoch [20/100], Loss: 31969.7539
Epoch [20/100], Loss: 24340.6699
Epoch [30/100], Loss: 28887.5039
Epoch [30/100], Loss: 32606.4336
Epoch [30/100], Loss: 35163.9766
Epoch [30/100], Loss: 27310.4648
Epoch [30/100], Loss: 27897.3691
Epoch [30/100], Loss: 25044.3203
Epoch [30/100], Loss: 29925.6016
Epoch [30/100], Loss: 9074.9717
Epoch [40/100], Loss: 34955.0938
Epoch [40/100], Loss: 31331.7402
Epoch [40/100], Loss: 26017.5078
Epoch [40/100], Loss: 29251.0742
Epoch [40/100], Loss: 33644.3633
Epoch [40/100], Loss: 29575.9785
Epoch [40/1

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:
:



In [ ]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 22122.12890625


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [ ]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,15.695701,219.0
1,14.650855,70.0
2,15.441466,202.0
3,18.041176,230.0
4,14.916197,111.0
...,...,...
84,13.678734,153.0
85,12.863558,98.0
86,11.711343,37.0
87,12.186400,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

# **Exercise #1 Solution**

In [ ]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop
epochs = 1000

for epoch in range(epochs):
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)

    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/1000], Loss: 23628.2012
Epoch [10/1000], Loss: 24102.9473
Epoch [10/1000], Loss: 22578.502
Epoch [10/1000], Loss: 21466.6465
Epoch [10/1000], Loss: 19625.9121
Epoch [10/1000], Loss: 29053.8359
Epoch [10/1000], Loss: 28368.1484
Epoch [10/1000], Loss: 32378.6152
Epoch [20/1000], Loss: 22054.1113
Epoch [20/1000], Loss: 23114.5645
Epoch [20/1000], Loss: 20547.002
Epoch [20/1000], Loss: 22787.1895
Epoch [20/1000], Loss: 21221.3047
Epoch [20/1000], Loss: 22847.8438
Epoch [20/1000], Loss: 25883.5664
Epoch [20/1000], Loss: 48023.918
Epoch [30/1000], Loss: 22890.1699
Epoch [30/1000], Loss: 15241.3223
Epoch [30/1000], Loss: 23712.5352
Epoch [30/1000], Loss: 20731.8418
Epoch [30/1000], Loss: 24921.5645
Epoch [30/1000], Loss: 23482.0312
Epoch [30/1000], Loss: 18676.584
Epoch [30/1000], Loss: 20970.9258
Epoch [40/1000], Loss: 20196.375
Epoch [40/1000], Loss: 15213.0869
Epoch [40/1000], Loss: 26746.3047
Epoch [40/1000], Loss: 22651.5
Epoch [40/1000], Loss: 15368.4092
Epoch [40/1000], Loss:

In [ ]:
model.eval()
mse_values_1000 = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        mse = criterion(outputs, targets)
        mse_values_1000.append(mse.item())

avg_mse_1000 = np.mean(mse_values_1000)
print(f"Previous Average MSE (100 epochs): {avg_mse}")
print(f"New Average MSE (1000 epochs): {avg_mse_1000}")

Previous Average MSE (100 epochs): 22122.12890625
New Average MSE (1000 epochs): 2898.4334716796875


This is a massive decrease in error, meaning the model predictions are now much closer to the actual values. The extra epochs allowed the network's weights and biases to converge much more effectively.

# **Exercise #2 Solution**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
import numpy as np

# 1. Load the dataset and prepare loaders
data = load_diabetes()
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

# 2. Define an improved architecture
class OptimalDiabetesModel(nn.Module):
    def __init__(self):
        super(OptimalDiabetesModel, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.layers(x)

# 3. Re-initialize model, loss, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
opt_model = OptimalDiabetesModel().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(opt_model.parameters(), lr=0.001)

# 4. Train for 1000 epochs
epochs = 1000
for epoch in range(epochs):
    opt_model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = opt_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    if (epoch+1) % 200 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

# 5. Evaluate on test set
opt_model.eval()
mse_values_opt = []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = opt_model(inputs)
        mse = criterion(outputs, targets)
        mse_values_opt.append(mse.item())

avg_mse_opt = np.mean(mse_values_opt)
print(f"\nPrevious Model MSE (1000 epochs): 2898.43")
print(f"New Larger Architecture MSE (1000 epochs): {avg_mse_opt:.2f}")

Epoch [200/1000], Loss: 1768.5801
Epoch [400/1000], Loss: 2826.4175
Epoch [600/1000], Loss: 2899.9451
Epoch [800/1000], Loss: 1561.9775
Epoch [1000/1000], Loss: 1874.5291

Previous Model MSE (1000 epochs): 2898.43
New Larger Architecture MSE (1000 epochs): 2780.90


By changing the architecture to a larger and deeper model, test MSE is reduced further:

*   Previous Model (with 1,000 epochs): Average MSE of 2,898.43
*   New Deeper Model (10 -> 64 -> 32 -> 16 -> 1 with ReLU): Average MSE of 2,780.90


